# Larval drift between Ogasawara island groups with OpenDrift

An exploratory **passive larval transport** notebook for the Hahajima, Chichijima,
and Mukojima groups in the Ogasawara (Bonin) archipelago, Japan.
It follows the ILIAD oil-spill example's setup → seed → run → plot structure and
its MPA example's daily releases → trajectories → connectivity workflow.

**Question:** Can waterborne particles released near one island group encounter
another group's neighbourhood during a specified larval age window?

This uses `OceanDrift`, matching the MPA reference. It does not use oil physics or
assume the cod-based egg, growth, and buoyancy defaults in `LarvalFish` apply to
Japanese fish. No species was specified. Dates, depth, duration, diffusion,
release points, and encounter circles below are **illustrative assumptions**.
They are not surveyed spawning sites, reef habitat, or protected-area boundaries.

**Spatial limit:** Copernicus global currents have a 1/12° (~8–9 km) grid.
That is suitable for an exploratory regional workflow, but cannot establish
transport through narrow channels, nearshore reef retention, or settlement.
A fine coastline mask does not improve the hydrodynamic grid's resolution.

## Before running

From this folder, create the environment using the accompanying `environment.yml`,
activate it, and launch JupyterLab (see `README.md`). Log in to Copernicus Marine
once using `copernicusmarine login`; do not put credentials in the notebook.
A compatible local CF NetCDF file can be used instead.

Run cells in order. A full default run releases 2,700 particles and tracks the
last cohort for 30 days. Results are generated only when the simulation runs;
no simulated scientific findings are supplied in advance.


### Copernicus account configuration

Enter your username and password directly in the two variables below, then run
this cell. The download cell uses them automatically. **These values are saved
in the notebook and visible to anyone you share it with.** The password is not
printed in outputs or included in the simulation results archive.

The default `CURRENT_SOURCE = 'edito_catalogue'` reads public catalogue-linked
current assets and requires no Copernicus password. These variables are only
needed if you choose the `copernicus` Toolbox download route.


In [ ]:
# Enter your Copernicus login below. These values are saved with the notebook.
COPERNICUS_USERNAME = ''  # enter your own username locally
COPERNICUS_PASSWORD = ''  # enter your own password locally; do not commit it

# The download cell uses these variables; no interactive login prompt is needed.
if COPERNICUS_USERNAME.strip() and COPERNICUS_PASSWORD:
    _copernicus_credentials = {
        'username': COPERNICUS_USERNAME.strip(),
        'password': COPERNICUS_PASSWORD,
    }
    print('Copernicus credentials configured for downloads.')
else:
    _copernicus_credentials = {}
    print('No Copernicus login configured. EDITO catalogue access and local files do not require it.')


In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import platform
import warnings
import uuid

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import folium
from IPython.display import display
from opendrift.models.oceandrift import OceanDrift
from opendrift.readers import reader_netCDF_CF_generic, reader_global_landmask

from drift_analysis import circle, release_cloud, encounter_matrix

ROOT = Path.cwd()
if not (ROOT / 'sites.csv').exists():
    raise RuntimeError('Open Jupyter from the folder containing this notebook and sites.csv.')
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
print('OpenDrift', version('opendrift'))


## 1. Configure an explicit scenario

All times are **UTC**. The May 2025 example is a reproducible historical scenario,
not a claim about spawning season. Change it once species and season are known.
Daily pulses over three days follow the MPA reference; increase `RELEASE_DAYS`
for a seasonal experiment. Every cohort receives the same maximum age.

`RELEASE_DEPTH_M` is positive downward, translated to OpenDrift's negative `z`.
The baseline holds depth constant: no vertical mixing, vertical advection,
swimming, growth, mortality, buoyancy, windage, or wave Stokes drift. Depth-specific
3D horizontal currents still vary through time and space. Compare depths in
separate runs; species-specific behaviour requires an additional calibrated model.

The 7–30 day encounter window is a sensitivity parameter, **not known settlement
competence**. Encounter zones are overlapping-capable, nonexclusive destinations.


In [ ]:
SCENARIO_LABEL = 'Ogasawara 30-day passive larval drift'
START = datetime(2025, 5, 1, 0, 0)
RELEASE_DAYS = 3
PARTICLES_PER_SITE_PER_DAY = 300
MAX_AGE_DAYS = 30
MIN_ENCOUNTER_AGE_DAYS = 7
RELEASE_DEPTH_M = 1.0
RELEASE_RADIUS_M = 1000.0
HORIZONTAL_DIFFUSIVITY_M2_S = 1.0  # assumed; compare 0, 1, 10, etc.
TIME_STEP_S = 600  # 10-minute integration
OUTPUT_STEP_S = 3600  # hourly saved output
RANDOM_SEED = 42

# Padded domain: widen if particles leave it; no zero-current fallback is used.
WEST, EAST, SOUTH, NORTH = 137.0, 148.0, 22.0, 33.0
# 6-hourly 3D horizontal currents; this is not the surface merged-wave current product.
DATASET_ID = 'cmems_mod_glo_phy-cur_anfc_0.083deg_PT6H-i'
# Set to a local CF-compliant file to bypass downloading. It must have uo, vo,
# time, depth (positive down), latitude, longitude on a regular geographic grid.
LOCAL_CURRENT_FILE = None
CURRENT_SOURCE = 'edito_catalogue'  # 'edito_catalogue' or 'copernicus'
assert CURRENT_SOURCE in {'edito_catalogue', 'copernicus'}

assert RELEASE_DAYS >= 1 and PARTICLES_PER_SITE_PER_DAY >= 1
assert 0 <= MIN_ENCOUNTER_AGE_DAYS <= MAX_AGE_DAYS
assert MAX_AGE_DAYS > 0
assert RELEASE_DEPTH_M >= 0 and RELEASE_RADIUS_M > 0
assert TIME_STEP_S > 0 and OUTPUT_STEP_S % TIME_STEP_S == 0
assert HORIZONTAL_DIFFUSIVITY_M2_S >= 0
RELEASE_TIMES = [START + timedelta(days=i) for i in range(RELEASE_DAYS)]
END = RELEASE_TIMES[-1] + timedelta(days=MAX_AGE_DAYS)
RUN_DIR = ROOT / 'results' / (datetime.now().strftime('%Y%m%dT%H%M%S') + '-' + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True)
TRAJECTORY_FILE = RUN_DIR / 'trajectories.nc'
print('UTC:', START, 'to', END)
print('Output:', RUN_DIR)


## 2. Define release locations and encounter neighbourhoods

`sites.csv` contains editable, approximate offshore release points east of each
island group and 15 km geodesic target circles around representative group
locations. The circles are a transparent proximity metric, **not habitat polygons**.
Replace these with surveyed spawning locations and habitat polygons for a
settlement study. The current metric addresses three main island groups in the
linked map region, not every island in the entire Ogasawara archipelago.

The global GSHHG landmask checks every sampled release point. Nearest-grid wet
current checks follow below. Neither check proves suitable larval habitat.


In [ ]:
sites = pd.read_csv(ROOT / 'sites.csv')
required_columns = ['name', 'release_lon', 'release_lat', 'target_lon', 'target_lat', 'target_radius_km']
assert set(required_columns).issubset(sites.columns)
assert sites.name.is_unique and len(sites) > 0
numeric = sites[required_columns[1:]].to_numpy(dtype=float)
assert np.isfinite(numeric).all() and (sites.target_radius_km > 0).all()
for lon_column, lat_column in [('release_lon', 'release_lat'), ('target_lon', 'target_lat')]:
    assert sites[lon_column].between(WEST, EAST).all()
    assert sites[lat_column].between(SOUTH, NORTH).all()
sites.to_csv(RUN_DIR / 'sites.csv', index=False)
display(sites)

land = reader_global_landmask.Reader()
rng = np.random.default_rng(RANDOM_SEED)
clouds = []
for source, site in enumerate(sites.itertuples(index=False)):
    for release_time in RELEASE_TIMES:
        lon, lat = release_cloud(site.release_lon, site.release_lat, RELEASE_RADIUS_M,
                                 PARTICLES_PER_SITE_PER_DAY, rng)
        on_land = land.get_variables(['land_binary_mask'], time=release_time,
                                    x=lon, y=lat, z=np.full(len(lon), -RELEASE_DEPTH_M))['land_binary_mask']
        if np.asarray(on_land).any():
            raise ValueError(f'{site.name}: release cloud overlaps land; edit sites.csv or radius.')
        if not ((lon >= WEST) & (lon <= EAST) & (lat >= SOUTH) & (lat <= NORTH)).all():
            raise ValueError('Release cloud exceeds the forcing domain.')
        clouds.append((source, release_time, lon, lat))

fig, ax = plt.subplots(figsize=(8, 9), subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent([141.7, 142.7, 26.2, 28.1])
reader_global_landmask.plot_land(ax, 141.7, 26.2, 142.7, 28.1, fast=True,
                                 crs_plot=ccrs.PlateCarree(), crs_lonlat=ccrs.PlateCarree())
for site in sites.itertuples(index=False):
    x, y = circle(site.target_lon, site.target_lat, site.target_radius_km)
    ax.plot(x, y, '--', label=site.name + ' encounter zone')
    ax.scatter(site.release_lon, site.release_lat, marker='x', color='black')
    ax.text(site.target_lon - 0.12, site.target_lat + 0.04, site.name, fontsize=8)
ax.gridlines(draw_labels=True)
ax.legend(loc='lower left', fontsize=8)
ax.set_title(f'{SCENARIO_LABEL}\nOffshore releases (×) and encounter zones')
fig.savefig(RUN_DIR / 'release_locations.png', dpi=160, bbox_inches='tight')
plt.show()


## 3. Obtain currents for Japan

Norkyst, used in the Norwegian oil example, does not cover Japan. Here we subset
Copernicus Marine's **Global Ocean Physics Analysis and Forecast** horizontal
currents (`uo`, `vo`), including the model levels bracketing release depth.
The request includes six-hour time padding on both ends. The cached filename
includes a hash of the request; changing dates, domain, or depth creates a new file.

A free Copernicus Marine account may be required. The catalogue and archive
coverage can change. If dates are unavailable, choose dates in the current
catalogue or supply a compatible local file. Download/authentication failures
stop the notebook; they never trigger fabricated currents.


In [ ]:
from edito_catalogue import download_edito_currents

request = dict(
    dataset_id=DATASET_ID, variables=['uo', 'vo'],
    minimum_longitude=WEST, maximum_longitude=EAST,
    minimum_latitude=SOUTH, maximum_latitude=NORTH,
    minimum_depth=0, maximum_depth=max(2.0, RELEASE_DEPTH_M + 1.0),
    start_datetime=(START - timedelta(hours=6)).isoformat(),
    end_datetime=(END + timedelta(hours=6)).isoformat(),
    coordinates_selection_method='outside',
)
request_hash = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()[:16]
if LOCAL_CURRENT_FILE is not None:
    CURRENT_FILE = Path(LOCAL_CURRENT_FILE).expanduser().resolve()
    if not CURRENT_FILE.is_file():
        raise FileNotFoundError(CURRENT_FILE)
elif CURRENT_SOURCE == 'edito_catalogue':
    CURRENT_FILE = DATA_DIR / f'currents_edito_{request_hash}.nc'
    if not CURRENT_FILE.exists():
        CURRENT_SOURCE_METADATA = download_edito_currents(request, CURRENT_FILE)
    else:
        metadata_file = CURRENT_FILE.with_suffix('.source.json')
        CURRENT_SOURCE_METADATA = json.loads(metadata_file.read_text()) if metadata_file.exists() else None
else:
    import copernicusmarine
    CURRENT_FILE = DATA_DIR / f'currents_{request_hash}.nc'
    if not CURRENT_FILE.exists():
        copernicusmarine.subset(**request, **globals().get('_copernicus_credentials', {}),
                               output_directory=str(DATA_DIR),
                               output_filename=CURRENT_FILE.name)
print('Current file:', CURRENT_FILE)


In [ ]:
# Recover the exact cached request if this cell is rerun after a kernel restart.
from pathlib import Path
from datetime import timedelta
import hashlib
import json

if not globals().get('CURRENT_FILE') or not Path(CURRENT_FILE).is_file():
    needed = ['DATA_DIR', 'DATASET_ID', 'WEST', 'EAST', 'SOUTH', 'NORTH', 'START', 'END', 'RELEASE_DEPTH_M']
    missing = [name for name in needed if name not in globals()]
    if missing:
        raise RuntimeError('Run the imports and scenario-settings cells first, then rerun this cell.')
    cached_request = dict(
        dataset_id=DATASET_ID, variables=['uo', 'vo'],
        minimum_longitude=WEST, maximum_longitude=EAST,
        minimum_latitude=SOUTH, maximum_latitude=NORTH,
        minimum_depth=0.494025, maximum_depth=max(2.0, RELEASE_DEPTH_M + 1.0),
        start_datetime=(START - timedelta(hours=6)).isoformat(),
        end_datetime=(END + timedelta(hours=6)).isoformat(),
        coordinates_selection_method='outside',
    )
    candidates = []
    for minimum_depth in [0.494025, 0]:
        candidate_request = dict(cached_request, minimum_depth=minimum_depth)
        candidate_hash = hashlib.sha256(json.dumps(candidate_request, sort_keys=True).encode()).hexdigest()[:16]
        cache_prefix = 'currents_edito' if globals().get('CURRENT_SOURCE') == 'edito_catalogue' else 'currents'
        candidates.append(DATA_DIR / f'{cache_prefix}_{candidate_hash}.nc')
    if globals().get('LOCAL_CURRENT_FILE') is not None:
        candidates.insert(0, Path(LOCAL_CURRENT_FILE).expanduser().resolve())
    CURRENT_FILE = next((path for path in candidates if path.is_file()), None)
    if CURRENT_FILE is None:
        raise RuntimeError('No cached current file matches this scenario. Run the current-data download cell above, then rerun this cell.')
    print('Recovered cached current file:', CURRENT_FILE)

currents = xr.open_dataset(CURRENT_FILE)
for name in ['uo', 'vo', 'time', 'depth', 'latitude', 'longitude']:
    if name not in currents:
        raise ValueError(f'Missing CF coordinate/variable: {name}')
for name in ['time', 'depth', 'latitude', 'longitude']:
    values = currents[name].values
    if currents[name].ndim != 1 or not np.all(values[1:] > values[:-1]):
        raise ValueError(f'{name} must be a strictly increasing 1D coordinate.')
for variable, standard_name in [('uo', 'eastward_sea_water_velocity'), ('vo', 'northward_sea_water_velocity')]:
    if currents[variable].attrs.get('standard_name') != standard_name:
        raise ValueError(f'{variable} must have CF standard_name={standard_name}')
    if currents[variable].attrs.get('units') not in ['m/s', 'm s-1', 'm s^-1']:
        raise ValueError(f'{variable} must be in metres per second.')
if currents.depth.attrs.get('positive', 'down') != 'down':
    raise ValueError('depth must be positive downward.')
if currents.time.values[0] > np.datetime64(START) or currents.time.values[-1] < np.datetime64(END):
    raise ValueError('Current file does not cover the full release + drift period.')
if not (float(currents.depth.min()) <= RELEASE_DEPTH_M <= float(currents.depth.max())):
    raise ValueError('Requested depth is not bracketed by the current data.')
for name, low, high in [('longitude', WEST, EAST), ('latitude', SOUTH, NORTH)]:
    if float(currents[name].min()) > low or float(currents[name].max()) < high:
        raise ValueError(f'Current file must cover the configured {name} bounds, including edges.')
# Check all sampled release positions against the nearest wet forcing cells over
# the complete run. This deliberately rejects partially missing release cells.
for source, release_time, lon, lat in clouds:
    sampled = currents[['uo', 'vo']].sel(
        longitude=xr.DataArray(lon, dims='sample'),
        latitude=xr.DataArray(lat, dims='sample'), method='nearest'
    ).interp(depth=RELEASE_DEPTH_M).sel(time=slice(START, END))
    if not np.isfinite(sampled.to_array().values).all():
        raise ValueError(f'{sites.name.iloc[source]}: missing/coastal current cells at release points; '
                         'move the illustrative release offshore or use a finer forcing product.')
display(currents[['uo', 'vo']])


## 4. Configure OpenDrift and release passive larvae

Each source has a stable integer `origin_marker`, preserved for the analysis.
There is no fallback for missing horizontal currents: missing data or leaving the
forcing domain causes loss of usable trajectories, reported below.

The coastline action `previous` follows the MPA reference: a proposed move onto
land returns the particle to its previous wet position. This is a numerical
boundary rule and does not represent settlement. It may exaggerate retention.
`max_age_seconds` retires each cohort with one integration step of numerical
grace so the last cohort can complete the final step. Analysis strictly excludes
positions older than `MAX_AGE_DAYS`, giving equal intended follow-up.


In [ ]:
np.random.seed(RANDOM_SEED)
o = OceanDrift(loglevel=20, logfile=str(RUN_DIR / 'opendrift.log'))
reader = reader_netCDF_CF_generic.Reader(str(CURRENT_FILE), name='Ogasawara currents')
o.add_reader(reader, variables=['x_sea_water_velocity', 'y_sea_water_velocity'])
o.add_reader(land)
o.set_config('environment:fallback:x_sea_water_velocity', None)
o.set_config('environment:fallback:y_sea_water_velocity', None)
o.set_config('general:coastline_action', 'previous')
o.set_config('drift:advection_scheme', 'runge-kutta4')
o.set_config('environment:fallback:horizontal_diffusivity', HORIZONTAL_DIFFUSIVITY_M2_S)
o.set_config('drift:vertical_mixing', False)
o.set_config('drift:vertical_advection', False)
o.set_config('drift:stokes_drift', False)
# OpenDrift increments age before moving. One-step retirement grace keeps the
# final cohort active through the last integration step; analysis still enforces MAX_AGE_DAYS.
o.set_config('drift:max_age_seconds', MAX_AGE_DAYS * 86400 + TIME_STEP_S)
o.set_config('seed:ocean_only', False)  # exact positions already checked; no relocation

manifest = []
for source, release_time, lon, lat in clouds:
    o.seed_elements(lon=lon, lat=lat, number=len(lon), radius=0,
                    z=-RELEASE_DEPTH_M, wind_drift_factor=0, terminal_velocity=0,
                    origin_marker=source, time=release_time)
    manifest.extend({'source_index': source, 'source': sites.name.iloc[source],
                     'release_time_utc': release_time.isoformat(), 'lon': float(x),
                     'lat': float(y), 'depth_m': RELEASE_DEPTH_M} for x, y in zip(lon, lat))
manifest = pd.DataFrame(manifest)
manifest.to_csv(RUN_DIR / 'release_manifest.csv', index=False)
released = manifest.groupby('source_index').size().reindex(range(len(sites))).to_numpy()
print(f'{len(manifest):,} particles in {len(clouds)} site/day pulses')


## 5. Run and retain provenance

This can take several minutes or longer. Follow progress in `opendrift.log`.
NetCDF trajectories, the exact release manifest, settings, package versions, and
a SHA-256 of the forcing file are saved in a unique results directory. Keep the
cached forcing file if exact reruns are needed; operational products can be revised.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

provenance = {
    'scenario_label': SCENARIO_LABEL,
    'model': 'OceanDrift passive fixed-depth larval transport',
    'forcing_source': 'local file' if LOCAL_CURRENT_FILE is not None else ('Copernicus Marine via EDITO Data Catalogue' if CURRENT_SOURCE == 'edito_catalogue' else DATASET_ID),
    'current_source_metadata': globals().get('CURRENT_SOURCE_METADATA') if LOCAL_CURRENT_FILE is None else None,
    'download_request': request if LOCAL_CURRENT_FILE is None else None,
    'current_file': str(CURRENT_FILE), 'current_file_sha256': sha256_file(CURRENT_FILE),
    'start_utc': START.isoformat(), 'end_utc': END.isoformat(),
    'release_days': RELEASE_DAYS, 'particles_per_site_per_day': PARTICLES_PER_SITE_PER_DAY,
    'max_age_days': MAX_AGE_DAYS, 'minimum_encounter_age_days': MIN_ENCOUNTER_AGE_DAYS,
    'release_depth_m': RELEASE_DEPTH_M, 'release_radius_m': RELEASE_RADIUS_M,
    'horizontal_diffusivity_m2_s': HORIZONTAL_DIFFUSIVITY_M2_S,
    'time_step_s': TIME_STEP_S, 'output_step_s': OUTPUT_STEP_S,
    'random_seed': RANDOM_SEED, 'coastline_action': 'previous',
    'retirement_age_seconds': MAX_AGE_DAYS * 86400 + TIME_STEP_S,
    'domain': [WEST, EAST, SOUTH, NORTH], 'sites': sites.to_dict(orient='records'),
    'python': platform.python_version(),
    'packages': {p: version(p) for p in ['opendrift', 'copernicusmarine', 'numpy',
                                       'xarray', 'pandas', 'pyproj', 'folium']},
}
(RUN_DIR / 'provenance.json').write_text(json.dumps(provenance, indent=2))
o.run(end_time=END, time_step=TIME_STEP_S, time_step_output=OUTPUT_STEP_S,
      outfile=str(TRAJECTORY_FILE), stop_on_error=True,
      export_variables=['lon', 'lat', 'z', 'status', 'origin_marker', 'age_seconds'])
print('Saved:', TRAJECTORY_FILE)


## 6. Inspect trajectories and data losses

Check statuses before interpreting an encounter matrix. Leaving the regional
domain or missing forcing can lower fractions artificially. Widen the downloaded
domain and rerun if losses occur. Early deactivation also censors follow-up.
Retirement at the configured maximum age is expected.


In [ ]:
trajectories = xr.open_dataset(TRAJECTORY_FILE)
if trajectories.sizes['trajectory'] != len(manifest):
    raise ValueError('Output trajectory count differs from scheduled release count.')
status_meanings = str(trajectories.status.attrs.get('flag_meanings', '')).split()
status_values = np.atleast_1d(trajectories.status.attrs.get('flag_values', np.arange(len(status_meanings))))
status_labels = {int(value): label for value, label in zip(status_values, status_meanings)}
quality_rows = []
for i in range(trajectories.sizes['trajectory']):
    t = trajectories.isel(trajectory=i)
    mask = np.isfinite(t.status.values)
    indices = np.flatnonzero(mask)
    if not len(indices):
        raise ValueError(f'Trajectory {i} has no status records.')
    j = indices[-1]
    state = int(t.status.values[j])
    quality_rows.append({'trajectory': int(t.trajectory.values),
                         'source_index': int(t.origin_marker.values[j]),
                         'last_status': status_labels.get(state, f'status_{state}'),
                         'last_age_days': float(t.age_seconds.values[j]) / 86400})
quality = pd.DataFrame(quality_rows)
quality.to_csv(RUN_DIR / 'trajectory_quality.csv', index=False)
display(quality.groupby(['source_index', 'last_status']).size().rename('particles').to_frame())
early = quality.last_age_days < MAX_AGE_DAYS - OUTPUT_STEP_S / 86400
if early.any():
    warnings.warn(f'{int(early.sum())} particles have truncated follow-up; inspect trajectory_quality.csv.')

# Display at most 100 evenly selected trajectories per source, every third output.
# The encounter matrices below still use all particles and every stored output.
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection': ccrs.PlateCarree()})
lon_values, lat_values = trajectories.lon.values, trajectories.lat.values
plot_status = trajectories.status.values
finite = np.isfinite(lon_values) & np.isfinite(lat_values)
plot_bounds = [float(np.nanmin(lon_values)) - 0.2, float(np.nanmax(lon_values)) + 0.2,
               float(np.nanmin(lat_values)) - 0.2, float(np.nanmax(lat_values)) + 0.2]
ax.set_extent(plot_bounds)
reader_global_landmask.plot_land(ax, plot_bounds[0], plot_bounds[2], plot_bounds[1], plot_bounds[3],
                                 fast=True, crs_plot=ccrs.PlateCarree(), crs_lonlat=ccrs.PlateCarree())
for source, name in enumerate(sites.name):
    color = ['#e76f51', '#277da8', '#7b2cbf'][source % 3]
    indices = quality.index[quality.source_index == source].to_numpy()
    selected = indices[np.linspace(0, len(indices)-1, min(100, len(indices)), dtype=int)]
    for index in selected:
        keep = finite[index] & (plot_status[index] == 0)
        ax.plot(lon_values[index, keep][::3], lat_values[index, keep][::3],
                color=color, linewidth=0.6, alpha=0.3)
    ax.plot([], [], color=color, label=name)
ax.gridlines(draw_labels=True)
ax.legend(loc='best')
ax.set_title(f'{SCENARIO_LABEL}\nDisplay subset: up to 100 trajectories per source')
fig.savefig(RUN_DIR / 'trajectories.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Calculate source → destination encounter fractions

Rows are release sources; columns are encounter destinations. A particle counts
once per destination if at least one **active output position** lies inside its
geodesic circle during the age window. Hourly point sampling can miss brief visits;
reduce `OUTPUT_STEP_S` to assess sensitivity. This does not use line-segment
intersection, irreversible settlement, mortality, habitat quality, or abundance.

Fractions divide by **all particles released at that source**, including particles
later lost from the domain. Diagonal entries mean return/continued presence after
the minimum age, not proof of self-recruitment. One particle may encounter several
destinations, so row sums need not equal one. Compare the quality report above.


In [ ]:
counts, fractions, events = encounter_matrix(
    trajectories, sites, released,
    min_age_days=MIN_ENCOUNTER_AGE_DAYS, max_age_days=MAX_AGE_DAYS,
)
counts.to_csv(RUN_DIR / 'encounter_counts.csv')
fractions.to_csv(RUN_DIR / 'encounter_fractions.csv')
events.to_csv(RUN_DIR / 'first_encounters.csv', index=False)
display(counts)
display(fractions)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(fractions.values, vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(len(sites)), sites.name, rotation=25, ha='right')
ax.set_yticks(range(len(sites)), sites.name)
ax.set_xlabel('Destination encounter zone')
ax.set_ylabel('Release source')
for i in range(len(sites)):
    for j in range(len(sites)):
        value = fractions.iloc[i, j]
        ax.text(j, i, f'{value:.1%}\n(n={counts.iloc[i, j]})', ha='center', va='center',
                color='black' if value > 0.55 else 'white')
fig.colorbar(image, ax=ax, label='Fraction of released particles')
ax.set_title(f'{SCENARIO_LABEL}\nPotential encounters at ages {MIN_ENCOUNTER_AGE_DAYS}–{MAX_AGE_DAYS} days')
fig.tight_layout()
fig.savefig(RUN_DIR / 'encounter_matrix.png', dpi=180)
plt.show()


## 8. Interactive map

This exports a Folium map with switchable source layers, approximate encounter
zones, and up to 100 trajectories per source. The map is subsampled for display;
the matrix above uses **all particles and all stored time steps**. HTML requires
internet access for map tiles and JavaScript assets.


The green overlay uses the **published Ogasawara – Ordinary Zone boundary** from
ProtectedSeas via HUB Ocean (CC BY 4.0). The coloured circles remain assumed encounter
neighbourhoods. A second matrix uses the published polygons, grouped into three
analytical island sectors by their nearest island-group centre. These are **not
three separate official MPAs**, and this one designation does not represent all
marine protection categories in the archipelago. The boundary snapshot has not
been validated against the historical simulation dates.


In [ ]:
"""Actual published protection polygons, grouped into analytical island sectors."""
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import shape, mapping, box
from shapely.ops import unary_union
from pyproj import Geod


def mpa_sectors(collection, sites, bounds=(141.5, 26.2, 143.0, 28.1)):
    """Retain whole intersecting components; group by nearest island-group centre.

    The sectors are derived analytical groups, not separate official MPAs.
    The published boundary coordinates are not simplified, buffered, or clipped.
    """
    region = box(*bounds)
    parts_by_site = [[] for _ in range(len(sites))]
    geod = Geod(ellps="WGS84")
    for feature in collection["features"]:
        geometry = shape(feature["geometry"])
        if not geometry.is_valid:
            raise ValueError("Published MPA geometry is invalid; inspect before analysis.")
        parts = list(geometry.geoms) if geometry.geom_type == "MultiPolygon" else [geometry]
        for part in parts:
            if not part.intersects(region):
                continue
            centre = part.representative_point()
            _, _, distance = geod.inv(np.full(len(sites), centre.x),
                                     np.full(len(sites), centre.y),
                                     sites.target_lon.to_numpy(), sites.target_lat.to_numpy())
            parts_by_site[int(np.argmin(distance))].append(part)
    if any(not parts for parts in parts_by_site):
        raise ValueError("At least one island group has no protected-area component.")
    geometries = [unary_union(parts) for parts in parts_by_site]
    features = [{"type": "Feature", "geometry": mapping(g),
                 "properties": {"sector": name + " — Ordinary Zone sector",
                                "parent_site": "Ogasawara - Ordinary Zone",
                                "components": len(parts),
                                "classification": "derived analytical sector; not a separate designation"}}
                for g, name, parts in zip(geometries, sites.name, parts_by_site)]
    return geometries, {"type": "FeatureCollection", "features": features}


def polygon_encounters(ds, sites, geometries, released, min_age_days, max_age_days):
    """Physical encounters with real polygons, using active sampled positions."""
    arrays = {name: ds[name].transpose("trajectory", "time").values
              for name in ["lon", "lat", "status", "age_seconds", "origin_marker"]}
    marker = arrays["origin_marker"]
    has_source = np.isfinite(marker)
    if not has_source.any(axis=1).all():
        raise ValueError("A trajectory has no source marker.")
    sources = marker[np.arange(len(marker)), has_source.argmax(axis=1)].astype(int)
    if (sources < 0).any() or (sources >= len(sites)).any():
        raise ValueError("Unknown source marker.")
    if not np.array_equal(np.bincount(sources, minlength=len(sites)), released):
        raise ValueError("Release denominators do not match trajectory sources.")
    age = arrays["age_seconds"] / 86400
    valid = (np.isfinite(arrays["lon"]) & np.isfinite(arrays["lat"])
             & (arrays["status"] == 0) & (age >= min_age_days) & (age <= max_age_days))
    counts = np.zeros((len(sites), len(geometries)), dtype=int)
    for dest, geometry in enumerate(geometries):
        shapely.prepare(geometry)
        hit = np.zeros(valid.shape, dtype=bool)
        hit[valid] = shapely.intersects_xy(geometry, arrays["lon"][valid], arrays["lat"][valid])
        counts[:, dest] = np.bincount(sources[hit.any(axis=1)], minlength=len(sites))
    names = sites.name.tolist()
    result = pd.DataFrame(counts, index=names, columns=[n + " MPA sector" for n in names])
    result.index.name = "release_source"
    return result, result.div(np.asarray(released), axis=0)


# Published ProtectedSeas geometry, served publicly by HUB Ocean (CC BY 4.0).
# One published designation; the three sectors below are analytical subdivisions.
from urllib.request import urlopen
MPA_DATASET_ID = '39e7bf52-8aff-46aa-b4b4-b82a22abc1b7'
MPA_SOURCE_URL = ('https://api.hubocean.earth/api/features/collections/'
                  + MPA_DATASET_ID + '/items?limit=10')
MPA_FILE = DATA_DIR / 'ogasawara_mpa_hubocean.geojson'
if not MPA_FILE.exists():
    with urlopen(MPA_SOURCE_URL, timeout=60) as response:
        payload = response.read()
    collection = json.loads(payload)
    if not collection.get('features'):
        raise ValueError('No published protected-area geometry was returned.')
    MPA_FILE.write_bytes(payload)
mpa_collection = json.loads(MPA_FILE.read_text())
mpa_geometries, mpa_sector_collection = mpa_sectors(mpa_collection, sites)
(RUN_DIR / 'published_ogasawara_mpa.geojson').write_text(json.dumps(mpa_collection))
(RUN_DIR / 'derived_mpa_sectors.geojson').write_text(json.dumps(mpa_sector_collection))
print('Published protection record:', mpa_collection['features'][0]['properties']['site_name'])
print('The three matrix destinations are derived island-group sectors, not three separate official MPAs.')
print('Age window:', MIN_ENCOUNTER_AGE_DAYS, 'to', MAX_AGE_DAYS, 'days')
mpa_counts, mpa_fractions = polygon_encounters(
    trajectories, sites, mpa_geometries, released,
    MIN_ENCOUNTER_AGE_DAYS, MAX_AGE_DAYS,
)
mpa_counts.to_csv(RUN_DIR / 'mpa_encounter_counts.csv')
mpa_fractions.to_csv(RUN_DIR / 'mpa_encounter_fractions.csv')
display(mpa_counts)
display(mpa_fractions)
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(mpa_fractions.values, vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(len(sites)), sites.name, rotation=25, ha='right')
ax.set_yticks(range(len(sites)), sites.name)
ax.set_xlabel('Destination sector of published Ordinary Zone')
ax.set_ylabel('Offshore release source')
for i in range(len(sites)):
    for j in range(len(sites)):
        value = mpa_fractions.iloc[i, j]
        ax.text(j, i, f'{value:.1%}\n(n={mpa_counts.iloc[i, j]})', ha='center', va='center',
                color='black' if value > 0.55 else 'white')
fig.colorbar(image, ax=ax, label='Fraction of all released particles')
ax.set_title(f'{SCENARIO_LABEL}\nPublished MPA polygon encounters, ages {MIN_ENCOUNTER_AGE_DAYS}–{MAX_AGE_DAYS} days')
fig.tight_layout()
fig.savefig(RUN_DIR / 'mpa_encounter_matrix.png', dpi=180)
plt.show()
provenance_path = RUN_DIR / 'provenance.json'
run_provenance = json.loads(provenance_path.read_text())
run_provenance['mpa_analysis'] = {
    'provider': 'ProtectedSeas via HUB Ocean', 'license': 'CC BY 4.0',
    'dataset_id': MPA_DATASET_ID, 'source_url': MPA_SOURCE_URL,
    'geometry_sha256': hashlib.sha256(MPA_FILE.read_bytes()).hexdigest(),
    'published_site': 'Ogasawara - Ordinary Zone',
    'sector_definition': 'whole polygon components intersecting 141.5,26.2,143,28.1; assigned to nearest island-group target centre',
    'sector_component_counts': [f['properties']['components'] for f in mpa_sector_collection['features']],
    'interpretation': 'physical encounters, not settlement; snapshot boundaries not validated historically',
}
provenance_path.write_text(json.dumps(run_provenance, indent=2))

m = folium.Map(location=[27.1, 142.2], zoom_start=7, tiles='Esri.WorldImagery')
colors = ['#e76f51', '#277da8', '#7b2cbf', '#2a9d8f']
for source, site in enumerate(sites.itertuples(index=False)):
    color = colors[source % len(colors)]
    layer = folium.FeatureGroup(name=site.name, show=True)
    x, y = circle(site.target_lon, site.target_lat, site.target_radius_km)
    folium.Polygon(list(zip(y, x)), color=color, weight=2, fill=True, fill_opacity=0.08,
                   tooltip=site.name + ': assumed encounter zone').add_to(layer)
    folium.Marker([site.release_lat, site.release_lon],
                  tooltip=site.name + ': illustrative offshore release').add_to(layer)
    source_indices = quality.index[quality.source_index == source].to_numpy()
    chosen = source_indices[np.linspace(0, len(source_indices) - 1,
                                       min(100, len(source_indices)), dtype=int)]
    for i in chosen:
        t = trajectories.isel(trajectory=int(i))
        mask = (np.isfinite(t.lon.values) & np.isfinite(t.lat.values)
                & (t.status.values == 0) & (t.age_seconds.values <= MAX_AGE_DAYS * 86400))
        locations = list(zip(t.lat.values[mask][::3], t.lon.values[mask][::3]))
        if len(locations) > 1:
            folium.PolyLine(locations, color=color, weight=1, opacity=0.25).add_to(layer)
    layer.add_to(m)
folium.GeoJson(
    mpa_collection,
    name='Published Ogasawara Ordinary Zone — ProtectedSeas / HUB Ocean',
    style_function=lambda feature: {'color': '#00ff88', 'weight': 2, 'fillColor': '#00ff88', 'fillOpacity': 0.12},
    tooltip=folium.GeoJsonTooltip(fields=['site_name', 'designation'], aliases=['Published site', 'Designation']),
).add_to(m)
m.fit_bounds([[sites.target_lat.min() - 0.25, sites.target_lon.min() - 0.25],
              [sites.target_lat.max() + 0.25, sites.target_lon.max() + 0.25]])
folium.LayerControl(collapsed=False).add_to(m)
m.save(str(RUN_DIR / 'interactive_trajectories.html'))
display(m)


## Interpretation and next experiments

1. Identify target fish species, spawning months, egg/larval depth distributions,
   pelagic duration, and competence window. Replace the illustrative values.
2. Compare release seasons, multiple years, depths, release cloud sizes, target
   radii, diffusion coefficients, timestep, and output interval. Repeat stochastic
   runs with several seeds; one scenario is not a connectivity climatology.
3. Obtain finer regional hydrodynamics (including relevant tides) before drawing
   conclusions about narrow passages or nearshore habitats. Validate currents and
   trajectories against observations where available.
4. Replace neighbourhood circles with reef/habitat polygons and releases with
   documented spawning sites. Add calibrated larval behaviour before estimating
   biological settlement or recruitment. The present particles do not consume,
   grow, die, hatch, or stop after encountering a target.
5. Compare appropriately dated MPA boundaries and human-pressure layers with
   trajectories as context. The HUB Ocean view's whale observations and fishing
   effort do not supply fish larval parameters or ocean currents. Its 2012–2024
   cumulative fishing layer is not a contemporaneous May 2025 measurement.

## References and data provenance

- [ILIAD oil-spill notebook](https://github.com/ILIAD-ocean-twin/opendrift-demo/blob/master/notebooks/oilspill.ipynb)
- [ILIAD MPA simulation notebook](https://github.com/ILIAD-ocean-twin/aquaculture-norway/blob/iliad-mpa/notebooks/mpa/1_opendrift-traj-sim-mpa.ipynb)
- [ILIAD MPA connectivity and visualization notebooks](https://github.com/ILIAD-ocean-twin/aquaculture-norway/tree/iliad-mpa/notebooks/mpa)
- [Copernicus global physics product and current dataset access](https://data.marine.copernicus.eu/product/GLOBAL_ANALYSISFORECAST_PHY_001_024/services)
- [Copernicus Marine Toolbox Python API](https://toolbox-docs.marine.copernicus.eu/en/stable/python-interface.html)
- [OpenDrift OceanDrift implementation](https://opendrift.github.io/_modules/opendrift/models/oceandrift.html)
- [OpenDrift LarvalFish implementation and cod defaults](https://opendrift.github.io/_modules/opendrift/models/larvalfish.html)
- [HUB Ocean Ogasawara protected-area context](https://app.hubocean.earth/catalog/dataset/39e7bf52-8aff-46aa-b4b4-b82a22abc1b7/ogasawara-marine-protected-area-)

Acknowledge **EU Copernicus Marine Service Information** when using its data and
follow the product's citation guidance. The Ordinary Zone boundary is downloaded from HUB Ocean and attributed to
ProtectedSeas (CC BY 4.0); its analytical island sectors retain the published boundary coordinates. Reference workflows informed the structure;
implementation here is purpose-built for this configurable experiment.


### EDITO / MyOcean NetCDF export

The final cell also exports **`edito_larval_density.nc`**, a regular WGS84
`time × latitude × longitude` grid with all-source and source-specific active
particle densities, plus total particle counts. Use the density layer for an
animated map; individual tracks remain in the native `trajectories.nc`.
The 0.1° bins are a visualization choice and add no detail to the current model.
Densities count equal-weight model particles per km², not measured larvae or
settlement probabilities. Retired/missing positions and ages over 30 days are
excluded; zero indicates no valid active particles, including in unmasked land
cells. Count conservation is checked before and after NetCDF writing.

In [EDITO MyOcean](https://my-ocean.dive.edito.eu/), use **Personal Catalogue →
Upload**, select `edito_larval_density.nc`, and choose `particle_density` or one
of the source-specific density layers. The 24-hour example passed viewer import on 7 November 2026; verify each
new export in your EDITO session. This private upload is separate from publishing a
shared dataset in EDITO's Data Catalogue.

Copy results to permanent EDITO S3 storage before stopping the notebook service;
its working files are temporary. See `README-EDITO.md` in the downloadable bundle.

If MyOcean displays a 0–0 automatic colour range, open the layer settings
and set Min = 0 and Max = 0.5 particles/km², then adjust as needed. The grid is
sparse; this display setting does not change the exported data.


In [ ]:
from edito_export import export_edito_density

# EDITO / MyOcean visualization export (snapshot density, not recruitment).
EXPORT_GRID_RESOLUTION_DEG = 0.1  # approximately 10 km; independent of simulation step size
edito_export = export_edito_density(
    trajectories, RUN_DIR / 'edito_larval_density.nc', sites.name.tolist(),
    [WEST, EAST, SOUTH, NORTH], resolution_deg=EXPORT_GRID_RESOLUTION_DEG,
    max_age_days=MAX_AGE_DAYS, depth_m=RELEASE_DEPTH_M,
)
(RUN_DIR / 'edito_export_checks.json').write_text(json.dumps(edito_export, indent=2))
print('Viewer grid:', edito_export['path'])
print('Grid:', edito_export['dimensions'], '| count conservation:', edito_export['count_conservation'])
print('In EDITO MyOcean: Personal Catalogue > Upload > select edito_larval_density.nc.')
print('Select particle_density or a source-specific density layer, then use the time slider.')
print('Native particle tracks remain in trajectories.nc. The 30-day EDITO run passed MyOcean import. For sparse grids, set the layer colour range to 0–0.5 particles/km² if Auto collapses to zero.')

